In [ ]:
!pip install pandas matplotlib gdown

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import gdown
import os

In [ ]:
# Download data from Google Sheets using gdown
sheet_id = '1e_lKct9ovnYByYkGpFCKrMpKQs5TQdKhTld9tDU1JcQ'
sheet_gid = '1499552579'
results_data = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'

csv_file = 'results_data.csv'

gdown.download(results_data, csv_file, quiet=False)

In [ ]:
# Load dataset
df = pd.read_csv(csv_file)
df

In [ ]:
# Clean up column names
df.columns = df.columns.astype(str).str.strip().str.lower().str.replace(' ', '_')
df['metaheuristic'] = df['metaheuristic'].replace({'ts1': 'ts'})
display(df)

In [ ]:
# Define standard styling
metaheuristics = df['metaheuristic'].unique()
colors = {
    'ts': '#9900FF',
    'ils': '#FE7033',
    'vns': '#32CD32'
}

# Remove rows with budgets of 6250 or 31250 from the DataFrame
df = df[~df['budget_ms'].isin([6250, 31250])]
# Map unique budgets to specific markers
unique_budgets = sorted(df['budget_ms'].dropna().unique())
available_markers = ['o', 's', '^', 'D', 'p', 'h', 'X']
budget_markers = {budget: available_markers[i % len(available_markers)] for i, budget in enumerate(unique_budgets)}

In [ ]:
# Generate the plots
objectives = df['obj_function'].unique()
graphs = df['graph'].unique()

for obj_func in objectives:
    for graph in graphs:

        plot_data = df[(df['obj_function'] == obj_func) & (df['graph'] == graph)].copy()
        if plot_data.empty:
            continue

        plot_data.sort_values(by=['metaheuristic', 'normalized_threads'], inplace=True)

        fig, ax = plt.subplots(figsize=(8, 6))

        ax.set_xlabel('Threads Ratio (SMAC3 / Standard)', fontsize=19)
        ax.set_ylabel('Score Ratio (SMAC3 / Standard)', fontsize=19)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.set_xlim(0, 1.2)

        #if(obj_func == "triangledensestsubgraph"):
        #    ax.set_ylim(0, 5.6)
        #else:
        #    ax.set_ylim(0, 2)
        ax.set_ylim(0, 2)

        ax.axhline(1.0, color='blue', linestyle='--', alpha=0.6, label='Score Standard')
        ax.axvline(1.0, color='red', linestyle='--', alpha=0.6, label='Thread Standard')

        # Plot each metaheuristic
        for meta in metaheuristics:
            meta_data = plot_data[plot_data['metaheuristic'] == meta]

            if not meta_data.empty:
                # Draw the continuous line for the Metaheuristic
                ax.plot(
                    meta_data['normalized_threads'],
                    meta_data['normalized_cost'],
                    color=colors.get(meta, 'black'),
                    linestyle='-',
                    linewidth=2.5,
                    alpha=0.9
                )

                # Loop through budgets and stamp the specific markers + error bars
                for budget in unique_budgets:
                    budget_data = meta_data[meta_data['budget_ms'] == budget]

                    if not budget_data.empty:
                        ax.errorbar(
                            budget_data['normalized_threads'],
                            budget_data['normalized_cost'],
                            yerr=budget_data['normalized_ci'],
                            color=colors.get(meta, 'black'),
                            marker=budget_markers[budget],
                            markersize=9,         # Slightly larger to make shapes clear
                            linestyle='none',     # NO LINE here, just the stamp
                            elinewidth=1.5,
                            capsize=2,
                            alpha=0.9
                        )

        ax.grid(True, which="major", ls=":", alpha=0.6)

        # Build a custom legend
        legend_elements = []
        for meta in metaheuristics:
            legend_elements.append(mlines.Line2D([0], [0], color=colors[meta], lw=2.5, label=f"SMAC3 - {meta.upper()}"))

        # Add a blank space
        legend_elements.append(mlines.Line2D([], [], color='none', label=' '))

        # Manually add the baselines
        legend_elements.append(mlines.Line2D([0], [0], color='blue', linestyle='--', alpha=0.6, label='Score Standard'))
        legend_elements.append(mlines.Line2D([0], [0], color='red', linestyle='--', alpha=0.6, label='Thread Standard'))

        # Add a blank space
        legend_elements.append(mlines.Line2D([], [], color='none', label=' '))

        # Add budgets to legend
        for budget in unique_budgets:
            b_int = int(budget)
            b_label = f"{b_int} ms"
            legend_elements.append(mlines.Line2D([0], [0], marker=budget_markers[budget], color='w',
                                                 markerfacecolor='black', markersize=9, label=f"{b_label}"))


        # Apply the custom legend
        ax.legend(handles=legend_elements, loc='best', fontsize=11, ncol=2)

        plt.tight_layout()

        # Save file
        output_folder = "pareto_plots/quality_threads"
        os.makedirs(output_folder, exist_ok=True)
        filename = f'quality_threads_{graph}_{obj_func}.pdf'
        filepath = os.path.join(output_folder, filename)
        plt.savefig(filepath, bbox_inches='tight')

        plt.show()